In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)                 # same random numbers on every run
tf.config.experimental.enable_op_determinism()  # same results on every run
DATA = Path("data")

## Convolution by hand
`conv2d` places the filter at every position where it fits, multiplies cell by cell and adds up.

In [ ]:
def conv2d(X, K):
    """Valid convolution (no padding, stride 1), as CNNs use it: no flipping of the filter."""
    n_r, n_c = X.shape[0] - K.shape[0] + 1, X.shape[1] - K.shape[1] + 1
    Z = np.zeros((n_r, n_c))
    for i in range(n_r):
        for j in range(n_c):
            # the window under the filter, times the filter, summed
            Z[i, j] = (X[i:i + K.shape[0], j:j + K.shape[1]] * K).sum()
    return Z

## The 6 x 6 example: a horizontal edge

In [ ]:
X = np.vstack([np.zeros((3, 6)), np.full((3, 6), 255)])   # top half black (0), bottom half white (255)
K_h = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])      # horizontal-edge filter
K_v = K_h.T                                                # vertical-edge filter
print(X.astype(int))
print("horizontal-edge filter:\n", conv2d(X, K_h).astype(int))
print("vertical-edge filter:\n", conv2d(X, K_v).astype(int))

## Keras' Conv2D gives the same numbers
We load our filter into a `Conv2D` layer by hand. Keras wants a batch axis and a channel axis.

In [ ]:
layer = keras.layers.Conv2D(1, 3, use_bias=False)
layer.build((None, 6, 6, 1))
layer.set_weights([K_h.reshape(3, 3, 1, 1).astype("float32")])
Zk = layer(X.reshape(1, 6, 6, 1).astype("float32")).numpy()[0, :, :, 0]
print(Zk.astype(int))
print("largest difference from NumPy:", np.abs(Zk - conv2d(X, K_h)).max())
# a true (flipped) convolution would give the opposite sign here
print("with the filter flipped:", conv2d(X, K_h[::-1, ::-1])[1, 0])

## Output size: n - f + 1

In [ ]:
rows = []
for n in (6, 28, 64):
    for f in (3, 5):
        shape = keras.layers.Conv2D(1, f)(np.zeros((1, n, n, 1), "float32")).shape
        rows.append(dict(n=n, f=f, formula=n - f + 1, keras=shape[1]))
print(pd.DataFrame(rows).to_string(index=False))

## Edges in a real photo
A 256 x 256 greyscale photo (SciPy's `ascent` image, public domain). Values 0 (black) to 255 (white).

In [ ]:
photo = np.array(Image.open(DATA / "photo.png"), dtype=float)
print(photo.shape, photo.min(), photo.max())
V = conv2d(photo, K_v)          # responds where brightness changes from left to right
H = conv2d(photo, K_h)          # responds where brightness changes from top to bottom
print("feature maps:", V.shape, H.shape)
# How strongly each filter answers on rows/columns: a vertical edge is a change along a row
print("mean |vertical map|  :", np.abs(V).mean().round(1))
print("mean |horizontal map|:", np.abs(H).mean().round(1))
np.savez_compressed(DATA / "photo_maps.npz", V=V.astype("float32"), H=H.astype("float32"))

## A left-edge filter on an MNIST digit, then ReLU

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
digit = x_train[np.where(y_train == 0)[0][0]] / 255.0     # first 0 of the training set, scaled to 0-1
K_left = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])     # bright on the left, dark on the right
L = conv2d(digit, K_left)
print("feature map shape:", L.shape)
print("positive cells:", (L > 0.1).sum(), " negative cells:", (L < -0.1).sum())
R = np.maximum(L, 0)                                        # ReLU: negatives become 0
print("after ReLU, smallest value:", R.min(), " largest:", R.max().round(2))
np.savez_compressed(DATA / "digit_maps.npz", digit=digit.astype("float32"), L=L.astype("float32"))

## A colour image: the filter has 3 channels too

In [ ]:
rng = np.random.default_rng(0)
img = rng.integers(0, 256, size=(6, 6, 3)).astype("float32")   # a 6 x 6 RGB image
W = rng.normal(size=(3, 3, 3)).astype("float32")                # one 3 x 3 x 3 filter: 27 numbers
by_hand = sum(conv2d(img[:, :, c], W[:, :, c]) for c in range(3))  # each channel, then add the three maps
layer = keras.layers.Conv2D(1, 3, use_bias=False)
layer.build((None, 6, 6, 3))
layer.set_weights([W.reshape(3, 3, 3, 1)])
out = layer(img[None]).numpy()
print("Keras output shape:", out.shape, " largest difference from hand:", np.abs(out[0, :, :, 0] - by_hand).max())
for k in (1, 2, 10):
    print(k, "filters ->", keras.layers.Conv2D(k, 3)(img[None]).shape)

## A small CNN learns its own filters
8 filters of 3 x 3 start random. One convolution layer, then Flatten and a softmax output; 2 epochs on MNIST.

In [ ]:
xtr, xte = x_train[..., None] / 255.0, x_test[..., None] / 255.0
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(8, 3, activation="relu"),
    keras.layers.Flatten(),
    keras.layers.Dense(10, activation="softmax")])
before = model.layers[0].get_weights()[0][:, :, 0, :].copy()      # the random start
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(xtr, y_train, epochs=2, batch_size=128, verbose=0)
after = model.layers[0].get_weights()[0][:, :, 0, :]
print("test accuracy:", round(model.evaluate(xte, y_test, verbose=0)[1], 4))
print("mean change of a filter value:", np.abs(after - before).mean().round(3))
maps = model.layers[0](digit[None, ..., None].astype("float32")).numpy()[0]   # the 8 feature maps of the digit
np.savez_compressed(DATA / "learned.npz", before=before, after=after, maps=maps)
for k in range(8):
    print(f"filter {k}:\n", after[:, :, k].round(2))